In [ ]:
import pandas, sklearn
print("ready")

In [ ]:
!git clone https://github.com/sybaik1/CodeComplex.git "D:/projects/datasets/CodeComplex"

In [ ]:
import os
for root, dirs, files in os.walk("D:/projects/datasets/CodeComplex"):
    if "'git" in root:
        continue
    print(root)
    for f in files[:15]:
        print("   ",f)


In [ ]:
from datasets import load_dataset
ds = load_dataset("google/code_x_glue_cc_defect_detection")
print(ds)
print(ds["train"].features)
print(ds['train'].to_pandas()["target"].value_counts())


In [ ]:
import pandas as pd
cx = pd.read_json("D:/projects/datasets/CodeComplex/data/python_data.jsonl", lines=True)
print(cx.shape)
print(cx.columns.tolist())
print(cx.head(3).T)
print(cx.select_dtypes("object").nunique())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

train = ds["train"].to_pandas()
test = ds["test"].to_pandas()

vec = TfidfVectorizer(token_pattern=r"[A-Za-z_]\w+|[^\sA-Za-z_0-9]",
                      max_features=50000, ngram_range=(1, 2))
Xtr = vec.fit_transform(train["func"])
Xte = vec.transform(test["func"])

clf = LogisticRegression(max_iter=1000).fit(Xtr, train["target"])
print(classification_report(test["target"], clf.predict(Xte)))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

train = ds["train"].to_pandas()
test = ds["test"].to_pandas()

vec = TfidfVectorizer(token_pattern=r"[A-Za-z_]\w+|[^\sA-Za-z_0-9]",
                      max_features=50000, ngram_range=(1, 2))
Xtr = vec.fit_transform(train["func"])
Xte = vec.transform(test["func"])

clf = LogisticRegression(max_iter=1000).fit(Xtr, train["target"])
print(classification_report(test["target"], clf.predict(Xte)))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

train = ds["train"].to_pandas()
test = ds["test"].to_pandas()

vec = TfidfVectorizer(token_pattern=r"[A-Za-z_]\w+|[^\sA-Za-z_0-9]",
                      max_features=50000, ngram_range=(1, 2))
Xtr = vec.fit_transform(train["func"])
Xte = vec.transform(test["func"])

clf = LogisticRegression(max_iter=1000).fit(Xtr, train["target"])
print(classification_report(test["target"], clf.predict(Xte)))

In [ ]:
import joblib, os
from sklearn.metrics import classification_report

os.makedirs("hajera/models", exist_ok=True)
os.makedirs("hajera/evaluation", exist_ok=True)

joblib.dump({"vectorizer": vec, "model": clf}, "hajera/models/defect_c_tfidf_lr.joblib")
open("hajera/evaluation/defect_c_report.txt", "w").write(
    classification_report(test["target"], clf.predict(Xte)))
print("saved")

In [ ]:
print(cx.shape)
print(cx.columns.tolist())
print(cx.head(2).T.astype(str).apply(lambda s: s.str[:60]))

In [ ]:
import ast, joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

def _nest(node, depth=0):
    best = depth
    for c in ast.iter_child_nodes(node):
        d = depth + 1 if isinstance(c, (ast.For, ast.While)) else depth
        best = max(best, _nest(c, d))
    return best

def extract_features(code):
    tree = ast.parse(code)
    nodes = list(ast.walk(tree))
    calls = [n for n in nodes if isinstance(n, ast.Call)]
    rec = 0
    for f in nodes:
        if isinstance(f, ast.FunctionDef):
            rec += sum(1 for n in ast.walk(f) if isinstance(n, ast.Call)
                       and isinstance(n.func, ast.Name) and n.func.id == f.name)
    bound = "unknown"
    for n in nodes:
        if isinstance(n, ast.For) and isinstance(n.iter, ast.Call) \
           and getattr(n.iter.func, "id", "") == "range":
            arg = n.iter.args[-1] if n.iter.args else None
            bound = "constant" if isinstance(arg, ast.Constant) else "input_dependent"
    return {
        "function_length": len(code.splitlines()),
        "loop_count": sum(isinstance(n, (ast.For, ast.While)) for n in nodes),
        "max_loop_nesting": _nest(tree),
        "branch_count": sum(isinstance(n, (ast.If, ast.IfExp)) for n in nodes),
        "recursion_count": rec,
        "function_call_count": len(calls),
        "ast_node_count": len(nodes),
        "input_size_variable": int(any(getattr(c.func, "id", "") in ("input", "len") for c in calls)),
        "loop_bound_type": bound,
        "uses_list": int(any(isinstance(n, ast.List) for n in nodes)),
        "uses_dict": int(any(isinstance(n, ast.Dict) for n in nodes)),
        "uses_set": int(any(isinstance(n, ast.Set) for n in nodes)),
    }

rows, skipped = [], 0
for _, r in cx.iterrows():
    try:
        f = extract_features(r["src"])
        f["label"], f["group"] = r["complexity"], r["problem"]
        rows.append(f)
    except Exception:
        skipped += 1
print("usable:", len(rows), "skipped:", skipped)

data = pd.get_dummies(pd.DataFrame(rows), columns=["loop_bound_type"])
X = data.drop(columns=["label", "group"]).astype(float)
y, groups = data["label"], data["group"]
print(y.value_counts())

tr, te = next(GroupShuffleSplit(test_size=0.2, random_state=42).split(X, y, groups))
cx_model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
cx_model.fit(X.iloc[tr], y.iloc[tr])

pred = cx_model.predict(X.iloc[te])
report = classification_report(y.iloc[te], pred)
print(report)
print(confusion_matrix(y.iloc[te], pred))

joblib.dump({"model": cx_model, "columns": list(X.columns)}, "hajera/models/complexity_python_rf.joblib")
open("hajera/evaluation/complexity_python_report.txt", "w").write(report)
print("saved")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import accuracy_score

src = cx["src"].reset_index(drop=True)
vec2 = TfidfVectorizer(token_pattern=r"[A-Za-z_]\w*|[^\sA-Za-z_0-9]",
                       ngram_range=(1, 3), max_features=80000, sublinear_tf=True)
Xt_tr = vec2.fit_transform(src.iloc[tr])
Xt_te = vec2.transform(src.iloc[te])

sc = StandardScaler().fit(X.iloc[tr])
A_tr = hstack([Xt_tr, csr_matrix(sc.transform(X.iloc[tr]))]).tocsr()
A_te = hstack([Xt_te, csr_matrix(sc.transform(X.iloc[te]))]).tocsr()

lr = LogisticRegression(max_iter=2000, C=5, class_weight="balanced").fit(A_tr, y.iloc[tr])
pred2 = lr.predict(A_te)
print(classification_report(y.iloc[te], pred2))

old, new = accuracy_score(y.iloc[te], pred), accuracy_score(y.iloc[te], pred2)
print("old:", round(old, 3), "new:", round(new, 3))
if new > old:
    joblib.dump({"vectorizer": vec2, "scaler": sc, "model": lr, "columns": list(X.columns)},
                "hajera/models/complexity_python_tfidf_lr.joblib")
    open("hajera/evaluation/complexity_python_tfidf_report.txt", "w").write(
        classification_report(y.iloc[te], pred2))
    print("saved better model")